In [523]:
import pandas as pd

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

### Feature Engineering

In [524]:
# Family Features
# total family members
# Create FamilySize (families survived more)
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

# is alone (1 = alone, 0 = not)
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

In [525]:
# title from Name
# Extract Title (captures age + social status)
train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

In [526]:
train["Title"].unique()

<StringArray>
[      'Mr',      'Mrs',     'Miss',   'Master',      'Don',      'Rev',
       'Dr',      'Mme',       'Ms',    'Major',     'Lady',      'Sir',
     'Mlle',      'Col',     'Capt', 'Countess', 'Jonkheer']
Length: 17, dtype: str

In [527]:
good_titles = ["Mr", "Miss", "Mrs", "Master"]

train["Title"] = train["Title"].apply(lambda x: x if x in good_titles else "Other")
test["Title"] = test["Title"].apply(lambda x: x if x in good_titles else "Other")

In [528]:
train["Title"].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64

In [529]:
# Fare Per Person
train["FarePerPerson"] = train["Fare"] / train["FamilySize"]
test["FarePerPerson"] = test["Fare"] / test["FamilySize"]

In [530]:
# use less columns
test_ids = test["PassengerId"]
train = train.drop(columns=["Name", "Ticket", "Cabin"])
test = test.drop(columns=["Name", "Ticket", "Cabin"])

In [531]:
train.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone,Title,FarePerPerson
0,1,0,3,male,22.0,1,0,7.2500,S,2,0,Mr,3.62500
1,2,1,1,female,38.0,1,0,71.2833,C,2,0,Mrs,35.64165
2,3,1,3,female,26.0,0,0,7.9250,S,1,1,Miss,7.92500
3,4,1,1,female,35.0,1,0,53.1000,S,2,0,Mrs,26.55000
4,5,0,3,male,35.0,0,0,8.0500,S,1,1,Mr,8.05000


### Encoding & Train Model

In [532]:
# convert categories columns to numbers
train = pd.get_dummies(train, columns=["Sex", "Embarked", "Title"], drop_first=True)
test = pd.get_dummies(test, columns=["Sex", "Embarked", "Title"], drop_first=True)

In [533]:
train.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Other
0,1,0,3,22.0,1,0,7.2500,2,0,3.62500,True,False,True,False,True,False,False
1,2,1,1,38.0,1,0,71.2833,2,0,35.64165,False,False,False,False,False,True,False
2,3,1,3,26.0,0,0,7.9250,1,1,7.92500,False,False,True,True,False,False,False
3,4,1,1,35.0,1,0,53.1000,2,0,26.55000,False,False,True,False,False,True,False
4,5,0,3,35.0,0,0,8.0500,1,1,8.05000,True,False,True,False,True,False,False


In [534]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [535]:
print(train.shape)
print(test.shape)

(891, 17)
(418, 17)


In [536]:
from sklearn.model_selection import train_test_split

# Prepare Data
# separate features and target
X = train.drop("Survived", axis=1)
y = train["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.fillna(X_train.median(numeric_only=True))
X_val = X_val.fillna(X_val.median(numeric_only=True))

test = test.fillna(test.median(numeric_only=True))

In [537]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(max_depth=8),
    # "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier()
}

In [538]:
best_model = None
best_score = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)

    print(name, ":", acc)

    if acc > best_score:
        best_score = acc
        best_model = model

print("Best model score:", best_score)

Random Forest : 0.8156424581005587
Gradient Boosting : 0.8100558659217877
KNN : 0.664804469273743
SVM : 0.5977653631284916
Decision Tree : 0.7374301675977654
Best model score: 0.8156424581005587


In [539]:
#Train best Model
best_model.fit(X, y)
test = test[X.columns]
predictions = best_model.predict(test)

In [540]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(best_model, X, y, cv=5)

print("Accuracy:", scores.mean())

Accuracy: 0.8215554579122465


In [541]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": predictions
})

submission.to_csv("../data/submission.csv", index=False)